In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
from typing import Dict, List, Tuple
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')


In [2]:
class StudentWellbeingAI:
    def __init__(self):
        self.data = {}
        self.merged_data = None
        self.student_profiles = {}
        self.risk_students = []
        self.recommendations = {}


In [3]:
def load_data(self):
    """Load all necessary CSV files"""
    try:
        # Essential data files
        file_mapping = {
            'student_checkins': './datasets/studentCheckin_cleaned.csv',
            'wellbeing_responses': './datasets/wellbeingResponse.csv', 
            'wellbeing_questions': './datasets/wellbeingQuestion.csv',
            'activities': './datasets/switches-summary.csv',
            'classes': './datasets/class.csv',
            'class_groups': './datasets/classGroup.csv', 
            'campus': './datasets/campus.csv'
        }
        
        loaded_files = []
        for key, filename in file_mapping.items():
            try:
                self.data[key] = pd.read_csv(filename)
                loaded_files.append(filename)
                print(f" Loaded {filename}: {self.data[key].shape[0]} rows, {self.data[key].shape[1]} columns")
            except FileNotFoundError:
                print(f" Warning: {filename} not found - skipping")
                self.data[key] = pd.DataFrame()  # Empty dataframe as placeholder
            except Exception as e:
                print(f" Warning: Error loading {filename}: {e}")
                self.data[key] = pd.DataFrame()
        
        if len(loaded_files) == 0:
            print(" No data files could be loaded")
            return False
        
        print(f"Successfully loaded {len(loaded_files)} data files")
        return True
        
    except Exception as e:
        print(f" Error loading data: {e}")
        return False

# Add method to class
StudentWellbeingAI.load_data = load_data


In [4]:
def clean_and_merge_data(self):
    """Step 1: Clean and merge student check-in data with wellbeing responses"""
    try:
        # Clean student check-ins
        checkins = self.data['student_checkins'].copy()
        
        print(f"Original checkins shape: {checkins.shape}")
        print(f"Checkins columns: {list(checkins.columns)}")
        
        # Process all data without sampling
        print(f"Processing all {len(checkins)} checkin records...")
        
        # Handle undefined values
        checkins = checkins.replace('undefined', np.nan)
        
        # Convert emotion intensity to numeric if column exists
        if 'Emotion Intensity Percentage' in checkins.columns:
            checkins['Emotion Intensity Percentage'] = pd.to_numeric(
                checkins['Emotion Intensity Percentage'], errors='coerce'
            )
        
        # Convert tiredness to numeric if column exists
        if 'Tiredness' in checkins.columns:
            checkins['Tiredness'] = pd.to_numeric(checkins['Tiredness'], errors='coerce')
        
        # Clean wellbeing responses (filter for students only if User Type column exists)
        wellbeing = self.data['wellbeing_responses'].copy()
        print(f"Original wellbeing shape: {wellbeing.shape}")
        
        # Process all wellbeing data without sampling
        print(f"Processing all {len(wellbeing)} wellbeing records...")
        
        print(f"Wellbeing columns: {list(wellbeing.columns)}")
        
        if 'User Type' in wellbeing.columns:
            wellbeing = wellbeing[wellbeing['User Type'] == 'STUDENT']
            print(f"Student wellbeing records: {len(wellbeing)}")
        
        # Convert timestamps
        print("Converting timestamps...")
        for col in ['Created At', 'Opened Date Time']:
            if col in wellbeing.columns:
                wellbeing[col] = pd.to_datetime(wellbeing[col], errors='coerce')
            if col in checkins.columns:
                checkins[col] = pd.to_datetime(checkins[col], errors='coerce')
        
        # Merge with wellbeing questions for context if both dataframes have required columns
        print("Merging wellbeing with questions...")
        if not wellbeing.empty and 'Wellbeing Question ID' in wellbeing.columns:
            questions = self.data['wellbeing_questions'].copy()
            if 'id' in questions.columns:
                wellbeing = wellbeing.merge(
                    questions, 
                    left_on='Wellbeing Question ID', 
                    right_on='id', 
                    how='left'
                )
                print(f"Wellbeing after questions merge: {wellbeing.shape}")
        
        # Create comprehensive student data
        print("Merging checkins with wellbeing data...")
        # Check if we have the necessary columns for merging
        if 'Student ID' in checkins.columns and 'User ID' in wellbeing.columns and not wellbeing.empty:
            self.merged_data = checkins.merge(
                wellbeing, 
                left_on='Student ID', 
                right_on='User ID', 
                how='left'
            )
            print(f"Merged data shape: {self.merged_data.shape}")
        else:
            # If we can't merge, just use checkins data
            print("⚠️ Warning: Using only check-in data (no wellbeing responses to merge)")
            self.merged_data = checkins
        
        # Process all merged data without sampling
        print(f"Processing all {len(self.merged_data)} merged records...")
        
        print(f" Data processed successfully. Final records: {len(self.merged_data)}")
        print(f"Final columns: {list(self.merged_data.columns)}")
        return True
        
    except Exception as e:
        print(f" Error in data cleaning/merging: {e}")
        import traceback
        traceback.print_exc()
        return False

# Add method to class
StudentWellbeingAI.clean_and_merge_data = clean_and_merge_data


In [5]:
def create_emotional_profiles(self):
    """Step 2: Create emotional profiles for each student over time"""
    try:
        profiles = {}
        
        # Debug: Check what columns we actually have
        print(f"Available columns: {list(self.merged_data.columns)}")
        print(f"Data shape: {self.merged_data.shape}")
        
        # Get unique students and sample if too many
        unique_students = self.merged_data['Student ID'].dropna().unique()
        print(f"Total unique students: {len(unique_students)}")
        
        print(f"Processing {len(unique_students)} students...")
        
        # Process students with progress tracking using tqdm
        for student_id in tqdm(unique_students, desc="Creating emotional profiles", unit="students"):
            
            if pd.isna(student_id):
                continue
                
            student_data = self.merged_data[self.merged_data['Student ID'] == student_id]
            
            if student_data.empty:
                continue
            
            # Process all student data without sampling
            
            # Safely get dominant emotion
            emotions = student_data['Emotion'].dropna()
            if not emotions.empty:
                emotion_mode = emotions.mode()
                dominant_emotion = emotion_mode.iloc[0] if len(emotion_mode) > 0 else 'Unknown'
            else:
                dominant_emotion = 'Unknown'
            
            # Aggregate emotional data
            profile = {
                'student_id': student_id,
                'total_checkins': len(student_data),
                'emotions': {
                    'dominant_emotion': dominant_emotion,
                    'avg_intensity': student_data['Emotion Intensity Percentage'].mean() if 'Emotion Intensity Percentage' in student_data.columns else None,
                    'emotion_distribution': emotions.value_counts().to_dict() if not emotions.empty else {},
                    'recent_emotions': list(emotions.tail(5)) if not emotions.empty else []
                },
                'wellness_indicators': {
                    'avg_tiredness': student_data['Tiredness'].mean() if 'Tiredness' in student_data.columns else None,
                    'chat_requests': student_data['Request For Chat'].notna().sum() if 'Request For Chat' in student_data.columns else 0,
                    'absence_rate': (student_data['Absencce'].notna().sum() / len(student_data)) if 'Absencce' in student_data.columns else 0
                },
                'wellbeing_scores': {},
                'trend_analysis': self._analyze_emotional_trends(student_data),
                'last_checkin': student_data['Created At'].max() if 'Created At' in student_data.columns else None
            }
            
            # Add wellbeing category scores safely
            if 'category' in student_data.columns and 'Answer' in student_data.columns:
                for category in ['CONNECTEDNESS', 'SOCIAL_IDENTITY']:
                    category_data = student_data[student_data['category'] == category]
                    if not category_data.empty and 'Answer' in category_data.columns:
                        answers = pd.to_numeric(category_data['Answer'], errors='coerce').dropna()
                        if not answers.empty:
                            profile['wellbeing_scores'][category] = answers.mean()
            
            profiles[student_id] = profile
        
        self.student_profiles = profiles
        print(f" Created emotional profiles for {len(profiles)} students")
        return True
        
    except Exception as e:
        print(f" Error creating emotional profiles: {e}")
        import traceback
        traceback.print_exc()
        return False

# Add method to class
StudentWellbeingAI.create_emotional_profiles = create_emotional_profiles


In [6]:
def _analyze_emotional_trends(self, student_data):
    """Analyze emotional trends for a student"""
    try:
        if 'Created At' not in student_data.columns or student_data.empty:
            return {'trend': 'insufficient_data'}
        
        # Sort by date
        student_data = student_data.sort_values('Created At')
        
        # Emotional intensity trend
        intensities = student_data['Emotion Intensity Percentage'].dropna()
        if len(intensities) < 2:
            return {'trend': 'insufficient_data'}
        
        # Simple trend analysis
        recent_avg = intensities.tail(3).mean()
        earlier_avg = intensities.head(3).mean()
        
        trend = 'stable'
        if recent_avg > earlier_avg + 0.2:
            trend = 'improving'
        elif recent_avg < earlier_avg - 0.2:
            trend = 'declining'
        
        return {
            'trend': trend,
            'recent_avg_intensity': recent_avg,
            'earlier_avg_intensity': earlier_avg,
            'volatility': intensities.std()
        }
    except:
        return {'trend': 'insufficient_data'}

# Add method to class
StudentWellbeingAI._analyze_emotional_trends = _analyze_emotional_trends


In [7]:
def identify_risk_indicators(self):
    """Step 3: Identify risk indicators and at-risk students"""
    try:
        risk_students = []
        
        for student_id, profile in tqdm(self.student_profiles.items(), desc="Identifying risk indicators", unit="students"):
            risk_score = 0
            risk_factors = []
            
            # Emotional risk factors
            avg_intensity = profile['emotions']['avg_intensity']
            if pd.notna(avg_intensity) and avg_intensity < 0.3:
                risk_score += 3
                risk_factors.append('Low emotional intensity')
            
            # Dominant negative emotions
            dominant_emotion = profile['emotions']['dominant_emotion']
            if dominant_emotion in ['SAD', 'ANGRY', 'ANXIOUS', 'FRUSTRATED']:
                risk_score += 2
                risk_factors.append(f'Predominantly {dominant_emotion.lower()}')
            
            # High tiredness
            avg_tiredness = profile['wellness_indicators']['avg_tiredness']
            if pd.notna(avg_tiredness) and avg_tiredness >= 4:
                risk_score += 2
                risk_factors.append('High tiredness levels')
            
            # Chat requests (seeking help)
            if profile['wellness_indicators']['chat_requests'] > 2:
                risk_score += 2
                risk_factors.append('Multiple chat requests')
            
            # High absence rate
            if profile['wellness_indicators']['absence_rate'] > 0.3:
                risk_score += 1
                risk_factors.append('High absence rate')
            
            # Declining trend
            if profile['trend_analysis']['trend'] == 'declining':
                risk_score += 2
                risk_factors.append('Declining emotional trend')
            
            # Low wellbeing scores
            for category, score in profile['wellbeing_scores'].items():
                if pd.notna(score) and score <= 2:
                    risk_score += 1
                    risk_factors.append(f'Low {category.lower()} score')
            
            # Classify risk level
            risk_level = 'Low'
            if risk_score >= 6:
                risk_level = 'High'
            elif risk_score >= 3:
                risk_level = 'Medium'
            
            if risk_score > 0:
                risk_students.append({
                    'student_id': student_id,
                    'risk_score': risk_score,
                    'risk_level': risk_level,
                    'risk_factors': risk_factors,
                    'priority': 'Immediate' if risk_score >= 6 else 'Monitor'
                })
        
        self.risk_students = sorted(risk_students, key=lambda x: x['risk_score'], reverse=True)
        print(f" Identified {len(risk_students)} students with risk indicators")
        return True
        
    except Exception as e:
        print(f" Error identifying risk indicators: {e}")
        return False

# Add method to class
StudentWellbeingAI.identify_risk_indicators = identify_risk_indicators


In [8]:
def map_interventions(self):
    """Step 4: Map current states to appropriate interventions"""
    try:
        activities = self.data['activities'].copy()
        
        # Create emotion-activity mapping
        intervention_mapping = {}
        
        for _, activity in tqdm(activities.iterrows(), desc="Mapping interventions", total=len(activities), unit="activities"):
            emotions = str(activity.get('Emotions', '')).lower()
            title = activity.get('Title', '')
            summary = activity.get('Summary', '')
            duration = activity.get('Duration', '')
            levels = str(activity.get('Levels', ''))
            
            # Map activities to emotions they can help with
            emotion_targets = []
            if 'sad' in emotions:
                emotion_targets.extend(['SAD', 'DOWN', 'DEPRESSED'])
            if 'happy' in emotions:
                emotion_targets.extend(['NEUTRAL', 'BORED'])
            if 'excited' in emotions:
                emotion_targets.extend(['TIRED', 'LOW_ENERGY'])
            if 'angry' in emotions:
                emotion_targets.extend(['ANGRY', 'FRUSTRATED'])
            
            activity_info = {
                'title': title,
                'summary': summary,
                'duration': duration,
                'levels': levels,
                'target_emotions': emotion_targets,
                'categories': activity.get('Categories', ''),
                'has_lesson_plan': activity.get('Has Lesson Plan', 'No') == 'Yes'
            }
            
            for emotion in emotion_targets:
                if emotion not in intervention_mapping:
                    intervention_mapping[emotion] = []
                intervention_mapping[emotion].append(activity_info)
        
        self.intervention_mapping = intervention_mapping
        print(f" Created intervention mapping for {len(intervention_mapping)} emotion types")
        return True
        
    except Exception as e:
        print(f" Error mapping interventions: {e}")
        return False

# Add method to class
StudentWellbeingAI.map_interventions = map_interventions


In [9]:
def generate_recommendations(self):
    """Step 5: Generate actionable recommendations for teachers"""
    try:
        recommendations = {}
        
        # Get class context
        class_context = self._get_class_context()
        
        for student_id, profile in tqdm(self.student_profiles.items(), desc="Generating recommendations", unit="students"):
            student_recommendations = {
                'student_id': student_id,
                'risk_level': 'Low',
                'immediate_actions': [],
                'suggested_activities': [],
                'monitoring_notes': [],
                'explanation': ''
            }
            
            # Check if student is at risk
            risk_student = next((rs for rs in self.risk_students if rs['student_id'] == student_id), None)
            if risk_student:
                student_recommendations['risk_level'] = risk_student['risk_level']
                student_recommendations['immediate_actions'] = self._get_immediate_actions(risk_student)
            
            # Get activity recommendations based on current emotional state
            dominant_emotion = profile['emotions']['dominant_emotion']
            activities = self._get_activity_recommendations(dominant_emotion, profile)
            student_recommendations['suggested_activities'] = activities
            
            # Generate monitoring notes
            monitoring_notes = self._generate_monitoring_notes(profile)
            student_recommendations['monitoring_notes'] = monitoring_notes
            
            # Create explanation
            explanation = self._create_explanation(profile, risk_student)
            student_recommendations['explanation'] = explanation
            
            recommendations[student_id] = student_recommendations
        
        self.recommendations = recommendations
        print(f"Generated recommendations for {len(recommendations)} students")
        return True
        
    except Exception as e:
        print(f"Error generating recommendations: {e}")
        return False

# Add method to class
StudentWellbeingAI.generate_recommendations = generate_recommendations


In [10]:
def _get_class_context(self):
    """Get class and campus context for students"""
    try:
        # This would merge class, group, and campus data
        # For now, returning basic structure
        return {}
    except:
        return {}

def _get_immediate_actions(self, risk_student):
    """Get immediate actions for at-risk students"""
    actions = []
    
    if risk_student['risk_level'] == 'High':
        actions.append("Consider immediate one-on-one check-in")
        actions.append("Alert school counselor if available")
    
    if 'Multiple chat requests' in risk_student['risk_factors']:
        actions.append("Student has requested to chat - prioritize connection")
    
    if 'High tiredness levels' in risk_student['risk_factors']:
        actions.append("Check if student needs break or rest time")
    
    if 'Declining emotional trend' in risk_student['risk_factors']:
        actions.append("Monitor closely over next few days")
    
    return actions

def _get_activity_recommendations(self, emotion, profile):
    """Get activity recommendations based on emotional state"""
    activities = []
    
    # Get activities for the dominant emotion
    if hasattr(self, 'intervention_mapping') and emotion in self.intervention_mapping:
        mapped_activities = self.intervention_mapping[emotion][:3]  # Top 3
        for activity in mapped_activities:
            activities.append({
                'title': activity['title'],
                'summary': activity['summary'],
                'duration': activity['duration'],
                'reason': f"Helps address {emotion.lower()} feelings"
            })
    
    # Add general wellbeing activities if low scores
    for category, score in profile['wellbeing_scores'].items():
        if pd.notna(score) and score <= 2:
            if category == 'CONNECTEDNESS':
                activities.append({
                    'title': 'Group Activity',
                    'summary': 'Facilitate peer interaction and connection',
                    'duration': '15-20 minutes',
                    'reason': 'Low connectedness score detected'
                })
    
    return activities[:5]  # Return top 5 recommendations

# Add methods to class
StudentWellbeingAI._get_class_context = _get_class_context
StudentWellbeingAI._get_immediate_actions = _get_immediate_actions
StudentWellbeingAI._get_activity_recommendations = _get_activity_recommendations


In [11]:
def _generate_monitoring_notes(self, profile):
    """Generate monitoring notes for teachers"""
    notes = []
    
    # Emotional pattern notes
    if len(profile['emotions']['recent_emotions']) > 0:
        recent = profile['emotions']['recent_emotions']
        notes.append(f"Recent emotional pattern: {' → '.join(recent[-3:])}")
    
    # Tiredness patterns
    avg_tiredness = profile['wellness_indicators']['avg_tiredness']
    if pd.notna(avg_tiredness):
        if avg_tiredness >= 4:
            notes.append(f"High tiredness levels (avg: {avg_tiredness:.1f}/5)")
        elif avg_tiredness <= 2:
            notes.append(f"Generally energetic (avg: {avg_tiredness:.1f}/5)")
    
    # Trend information
    trend = profile['trend_analysis']['trend']
    if trend == 'improving':
        notes.append("Emotional wellbeing trending upward ✓")
    elif trend == 'declining':
        notes.append("Emotional wellbeing trending downward - monitor closely")
    
    return notes

def _create_explanation(self, profile, risk_student):
    """Create AI explanation for recommendations"""
    explanation = "AI Analysis: "
    
    # Base emotional state
    dominant_emotion = profile['emotions']['dominant_emotion']
    avg_intensity = profile['emotions']['avg_intensity']
    
    explanation += f"Student's dominant emotion is {dominant_emotion}"
    
    if pd.notna(avg_intensity):
        if avg_intensity < 0.3:
            explanation += " with low intensity, suggesting possible emotional challenges. "
        elif avg_intensity > 0.7:
            explanation += " with high intensity, indicating strong emotional engagement. "
        else:
            explanation += " with moderate intensity. "
    
    # Risk factors
    if risk_student:
        explanation += f"Risk level: {risk_student['risk_level']} due to: {', '.join(risk_student['risk_factors'][:2])}. "
    
    # Trend
    trend = profile['trend_analysis']['trend']
    if trend == 'declining':
        explanation += "Recent emotional trend shows decline - early intervention recommended."
    elif trend == 'improving':
        explanation += "Positive emotional trend detected - continue current support strategies."
    
    return explanation

# Add methods to class
StudentWellbeingAI._generate_monitoring_notes = _generate_monitoring_notes
StudentWellbeingAI._create_explanation = _create_explanation


In [12]:
def generate_teacher_dashboard(self):
    """Generate comprehensive teacher dashboard data"""
    try:
        dashboard = {
            'overview': {
                'total_students': len(self.student_profiles),
                'high_risk_count': len([r for r in self.risk_students if r['risk_level'] == 'High']),
                'medium_risk_count': len([r for r in self.risk_students if r['risk_level'] == 'Medium']),
                'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            },
            'priority_students': [
                {
                    'student_id': r['student_id'],
                    'risk_level': r['risk_level'],
                    'priority': r['priority'],
                    'key_concerns': r['risk_factors'][:2],
                    'recommendations': self.recommendations.get(r['student_id'], {}).get('immediate_actions', [])[:2]
                }
                for r in self.risk_students[:10]  # Top 10 priority students
            ],
            'class_emotional_summary': self._generate_class_summary(),
            'recommended_activities': self._get_top_recommended_activities()
        }
        
        return dashboard
        
    except Exception as e:
        print(f"Error generating dashboard: {e}")
        return {}

def _generate_class_summary(self):
    """Generate class-wide emotional summary"""
    try:
        emotions = []
        intensities = []
        
        for profile in self.student_profiles.values():
            if profile['emotions']['dominant_emotion'] != 'Unknown':
                emotions.append(profile['emotions']['dominant_emotion'])
            if pd.notna(profile['emotions']['avg_intensity']):
                intensities.append(profile['emotions']['avg_intensity'])
        
        summary = {
            'dominant_class_emotion': max(set(emotions), key=emotions.count) if emotions else 'Unknown',
            'average_emotional_intensity': np.mean(intensities) if intensities else 0,
            'emotional_distribution': {emotion: emotions.count(emotion) for emotion in set(emotions)}
        }
        
        return summary
    except:
        return {}

def _get_top_recommended_activities(self):
    """Get most frequently recommended activities"""
    try:
        activity_counts = {}
        
        for rec in self.recommendations.values():
            for activity in rec.get('suggested_activities', []):
                title = activity['title']
                if title not in activity_counts:
                    activity_counts[title] = {
                        'count': 0,
                        'summary': activity['summary'],
                        'duration': activity['duration']
                    }
                activity_counts[title]['count'] += 1
        
        # Sort by frequency
        top_activities = sorted(activity_counts.items(), key=lambda x: x[1]['count'], reverse=True)
        
        return [
            {
                'title': title,
                'recommended_for': data['count'],
                'summary': data['summary'],
                'duration': data['duration']
            }
            for title, data in top_activities[:5]
        ]
    except:
        return []

# Add methods to class
StudentWellbeingAI.generate_teacher_dashboard = generate_teacher_dashboard
StudentWellbeingAI._generate_class_summary = _generate_class_summary
StudentWellbeingAI._get_top_recommended_activities = _get_top_recommended_activities


In [ ]:
def run_complete_analysis(self):
    """Run the complete analysis pipeline"""
    print("Starting Student Wellbeing AI Analysis...")
    print("=" * 50)
    
    steps = [
        ("Loading data files", self.load_data),
        ("Cleaning and merging data", self.clean_and_merge_data),
        ("Creating emotional profiles", self.create_emotional_profiles),
        ("Identifying risk indicators", self.identify_risk_indicators),
        ("Mapping interventions", self.map_interventions),
        ("Generating recommendations", self.generate_recommendations)
    ]
    
    for step_name, step_function in steps:
        print(f"\n{step_name}...")
        if not step_function():
            print(f"Failed at: {step_name}")
            return False
    
    print("\n" + "=" * 50)
    print("  Analysis Complete!")
    
    # Generate and display dashboard
    dashboard = self.generate_teacher_dashboard()
    self.display_dashboard(dashboard)
    
    return True

def display_dashboard(self, dashboard):
    """Display teacher dashboard in readable format"""
    print("\n" + " TEACHER DASHBOARD" + "\n" + "=" * 50)
    
    # Overview
    overview = dashboard['overview']
    print(f"OVERVIEW:")
    print(f"   Total Students: {overview['total_students']}")
    print(f"   High Risk Students: {overview['high_risk_count']}")
    print(f"   Medium Risk Students: {overview['medium_risk_count']}")
    print(f"   Last Updated: {overview['last_updated']}")
    
    # Priority students
    print(f"\nPRIORITY STUDENTS:")
    for student in dashboard['priority_students'][:5]:
        print(f"   Student: {student['student_id'][:8]}...")
        print(f"   Risk Level: {student['risk_level']} ({student['priority']})")
        print(f"   Concerns: {', '.join(student['key_concerns'])}")
        print(f"   Actions: {', '.join(student['recommendations'])}")
        print("   " + "-" * 40)
    
    # Class summary
    class_summary = dashboard['class_emotional_summary']
    if class_summary:
        print(f"\nCLASS EMOTIONAL SUMMARY:")
        print(f"   Dominant Emotion: {class_summary.get('dominant_class_emotion', 'Unknown')}")
        print(f"   Avg Intensity: {class_summary.get('average_emotional_intensity', 0):.2f}")
    
    # Top activities
    print(f"\n TOP RECOMMENDED ACTIVITIES:")
    for activity in dashboard['recommended_activities'][:3]:
        print(f"   • {activity['title']} (for {activity['recommended_for']} students)")
        print(f"     {activity['summary']} | Duration: {activity['duration']}")

def export_results(self, filename='wellbeing_analysis_results.json'):
    """Export all results to JSON file"""
    try:
        results = {
            'student_profiles': self.student_profiles,
            'risk_students': self.risk_students,
            'recommendations': self.recommendations,
            'dashboard': self.generate_teacher_dashboard(),
            'export_timestamp': datetime.now().isoformat()
        }
        
        with open(filename, 'w') as f:
            json.dump(results, f, indent=2, default=str)
        
        print(f"Results exported to {filename}")
        return True
    except Exception as e:
        print(f"Error exporting results: {e}")
        return False

# Add methods to class
StudentWellbeingAI.run_complete_analysis = run_complete_analysis
StudentWellbeingAI.display_dashboard = display_dashboard
StudentWellbeingAI.export_results = export_results


In [14]:
# Main execution
if __name__ == "__main__":
    # Initialize the AI system
    ai_system = StudentWellbeingAI()
    
    # Run complete analysis
    success = ai_system.run_complete_analysis()
    
    if success:
        # Export results
        ai_system.export_results()
        
        print("\n" + " SUCCESSFULLY ANALYZED!" + "\n" + "=" * 50)
        print("The AI system has successfully analyzed student wellbeing data.")
        print("Teachers now have actionable insights and recommendations!")
        print("\nNext steps:")
        print("1. Review priority students requiring immediate attention")
        print("2. Implement suggested activities in classroom")
        print("3. Monitor student progress over time")
        print("4. Provide feedback to improve AI recommendations")
    else:
        print("\n Analysis failed. Please check your data files and try again.")


Starting Student Wellbeing AI Analysis...

Loading data files...
 Loaded ./datasets/studentCheckin_cleaned.csv: 757112 rows, 10 columns
 Loaded ./datasets/wellbeingResponse.csv: 477015 rows, 6 columns
 Loaded ./datasets/wellbeingQuestion.csv: 24 rows, 5 columns
 Loaded ./datasets/switches-summary.csv: 107 rows, 13 columns
 Loaded ./datasets/class.csv: 583 rows, 3 columns
 Loaded ./datasets/classGroup.csv: 7 rows, 3 columns
 Loaded ./datasets/campus.csv: 4 rows, 2 columns
Successfully loaded 7 data files

Cleaning and merging data...
Original checkins shape: (757112, 10)
Checkins columns: ['Check In Session ID', 'Student Check In ID', 'Student ID', 'Absencce', 'Emotion', 'Emotion Intensity Percentage', 'Tiredness', 'Request For Chat', 'Chat Cleared', 'Eating']
Processing all 757112 checkin records...
Original wellbeing shape: (477015, 6)
Processing all 477015 wellbeing records...
Wellbeing columns: ['Wellbeing Response ID', 'Created At', 'User Type', 'User ID', 'Wellbeing Question ID', 

Creating emotional profiles: 100%|██████████| 11669/11669 [7:41:14<00:00,  2.37s/students]      


 Created emotional profiles for 11669 students

Identifying risk indicators...


Identifying risk indicators: 100%|██████████| 11669/11669 [00:00<00:00, 203582.77students/s]


 Identified 9363 students with risk indicators

Mapping interventions...


Mapping interventions: 100%|██████████| 107/107 [00:00<00:00, 11990.45activities/s]


 Created intervention mapping for 9 emotion types

Generating recommendations...


Generating recommendations: 100%|██████████| 11669/11669 [00:04<00:00, 2898.52students/s]


Generated recommendations for 11669 students

 Analysis Complete!

 TEACHER DASHBOARD
OVERVIEW:
   Total Students: 11669
   High Risk Students: 309
   Medium Risk Students: 2817
   Last Updated: 2025-10-01 07:50:04

PRIORITY STUDENTS:
   Student: c50e6ae7...
   Risk Level: High (Immediate)
   Concerns: Low emotional intensity, Predominantly sad
   Actions: Consider immediate one-on-one check-in, Alert school counselor if available
   ----------------------------------------
   Student: 9bed8cdd...
   Risk Level: High (Immediate)
   Concerns: Low emotional intensity, High tiredness levels
   Actions: Consider immediate one-on-one check-in, Alert school counselor if available
   ----------------------------------------
   Student: be4fb349...
   Risk Level: High (Immediate)
   Concerns: Low emotional intensity, High tiredness levels
   Actions: Consider immediate one-on-one check-in, Alert school counselor if available
   ----------------------------------------
   Student: 9d03567c...
 